# Cafe Sales Analysis

Clean the transaction data, examine sales patterns, and compare revenue across items and payment methods.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load cafe_sales.csv from the project data directory.
data = pd.read_csv("cafe_sales.csv")

In [ ]:
# Preview the imported sales records.
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [5]:
# Replace UNKNOWN and ERROR with NaN
data = data.replace(["UNKNOWN","ERROR"], np.nan)

In [ ]:
# Coerce invalid numeric entries so they can be handled as missing values.
cols = ["Quantity", "Total Spent", "Price Per Unit"]
data[cols] = data[cols].apply(pd.to_numeric, errors="coerce").astype("float64")

In [8]:
data["Payment Method"].value_counts(dropna=False)

Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

In [ ]:
# Parse dates once so later monthly and weekly summaries can use datetime accessors.
data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], errors="coerce")

In [ ]:
# Fill missing numeric values when the other fields allow it.
data["Total Spent"] = data["Total Spent"].fillna(data["Quantity"] * data["Price Per Unit"])
data["Price Per Unit"] = data["Price Per Unit"].fillna(data["Total Spent"] / data["Quantity"])
data["Quantity"] = data["Quantity"].fillna(data["Total Spent"] / data["Price Per Unit"])

In [13]:
# drop rows where Quantity, Total Spent and Price Per Unit are all missing
data.dropna(subset=cols, how="all", inplace=True)


In [ ]:
# Keep rows with missing item labels under a consistent category.
data["Item"] = data["Item"].fillna("Unknown")

C:\Users\lefte\AppData\Local\Temp\ipykernel_11324\1426144868.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data["Item"] = data["Item"].fillna("Unknown", inplace=True)


In [15]:
data.dropna(subset=["Transaction Date"], inplace=True)

In [ ]:
# Preview the cleaned records.
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9993,TXN_4766549,Smoothie,2.0,4.0,8.0,Cash,NaN,2023-10-20
9994,TXN_7851634,Unknown,4.0,4.0,16.0,NaN,NaN,2023-01-08
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,NaN,2023-08-30
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02


In [17]:
data.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2704
Location            3407
Transaction Date       0
dtype: int64

In [ ]:
# Mark missing location and payment details as unknown.
col2 = ["Location", "Payment Method"]
data[col2] = data[col2].fillna("Unknown")

C:\Users\lefte\AppData\Local\Temp\ipykernel_11324\2779490604.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data[col2] = data[col2].fillna("Unknown", inplace=True)


In [19]:
data.describe().round(2)

,Quantity,Price Per Unit,Total Spent
count,8596.00,8596.00,8596.00
mean,3.02,2.95,8.92
std,1.42,1.28,5.99
min,1.00,1.00,1.00
25%,2.00,2.00,4.00
50%,3.00,3.00,8.00
75%,4.00,4.00,12.00
max,5.00,5.00,25.00


## Sales Patterns and Segments

In [20]:
# Which item earns the most revenue?

revenue = data.groupby("Item")["Total Spent"].sum().sort_values(ascending=False)
revenue

Item
Salad       14890.0
Sandwich    11680.0
Smoothie    11608.0
Juice        9108.0
Cake         9009.0
Unknown      7231.5
Coffee       6210.0
Tea          4179.0
Cookie       2775.0
Name: Total Spent, dtype: float64

In [21]:
revenubylocation = data.groupby("Location")["Total Spent"].sum().sort_values(ascending=False)
revenubylocation

Location
Unknown     30475.0
In-store    23428.5
Takeaway    22787.0
Name: Total Spent, dtype: float64

In [22]:
revenuebydate = data.groupby("Transaction Date")["Total Spent"].sum().sort_values(ascending=False)
revenuebydate

Transaction Date
2023-06-16    351.0
2023-11-07    349.5
2023-07-24    340.5
2023-08-26    331.0
2023-10-19    326.5
              ...  
2023-07-22     98.5
2023-11-05     95.0
2023-10-05     93.5
2023-07-30     89.0
2023-08-08     75.5
Name: Total Spent, Length: 365, dtype: float64

In [25]:
big_spent = data[data["Total Spent"] > 20]
big_spent.tail(3)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9879,TXN_6393305,Salad,5.0,5.0,25.0,Digital Wallet,In-store,2023-10-19
9908,TXN_8922585,Salad,5.0,5.0,25.0,Credit Card,Unknown,2023-06-22
9971,TXN_6120851,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-02-04


In [ ]:
# Find coffee and salad transactions above $5.
high_value_items = data[
    data["Item"].isin(["Salad", "Coffee"]) & (data["Total Spent"] > 5)
]
high_value_items

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
15,TXN_2847255,Salad,3.0,5.0,15.0,Credit Card,In-store,2023-11-15
37,TXN_1080432,Salad,2.0,5.0,10.0,Credit Card,In-store,2023-04-29
39,TXN_6688524,Coffee,4.0,2.0,8.0,Unknown,Unknown,2023-06-29
...,...,...,...,...,...,...,...,...
9954,TXN_1191659,Coffee,4.0,2.0,8.0,Credit Card,In-store,2023-11-21
9971,TXN_6120851,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-02-04
9973,TXN_7936002,Salad,2.0,5.0,10.0,Digital Wallet,Unknown,2023-01-26
9990,TXN_1538510,Coffee,5.0,2.0,10.0,Digital Wallet,Unknown,2023-05-22


In [29]:
not_takeaway = data[~(data["Location"] == "Takeaway")]
not_takeaway

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unknown,2023-03-31
...,...,...,...,...,...,...,...,...
9993,TXN_4766549,Smoothie,2.0,4.0,8.0,Cash,Unknown,2023-10-20
9994,TXN_7851634,Unknown,4.0,4.0,16.0,Unknown,Unknown,2023-01-08
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02


In [30]:
data["Item"].value_counts()

Item
Juice       1020
Coffee      1012
Cake         989
Salad        985
Sandwich     969
Smoothie     951
Cookie       929
Tea          918
Unknown      823
Name: count, dtype: int64

In [31]:
drinks = data[data["Item"].isin(["Juice", "Coffee", "Tea","Smoothie"])]

In [32]:
mid_category_spend = data[data["Total Spent"].between(5,15)]
mid_category_spend

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
6,TXN_4433211,Unknown,3.0,3.0,9.0,Unknown,Takeaway,2023-10-06
8,TXN_4717867,Unknown,5.0,3.0,15.0,Unknown,Takeaway,2023-07-28
12,TXN_7619095,Sandwich,2.0,4.0,8.0,Cash,In-store,2023-05-03
...,...,...,...,...,...,...,...,...
9990,TXN_1538510,Coffee,5.0,2.0,10.0,Digital Wallet,Unknown,2023-05-22
9991,TXN_3897619,Sandwich,3.0,4.0,12.0,Cash,Takeaway,2023-02-24
9993,TXN_4766549,Smoothie,2.0,4.0,8.0,Cash,Unknown,2023-10-20
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02


In [ ]:
# Group transaction values into low, medium, and high spend bands.
conditions = [
    data["Total Spent"] < 5,
    data["Total Spent"].between(5, 15),
    data["Total Spent"] > 15,
]
choices = ["Low", "Medium", "High"]
data["spend_band"] = np.select(conditions, choices, default="Unknown")

In [ ]:
data["spend_band"].value_counts(dropna=False)

0          Low
1       Medium
2          Low
3       Medium
4          Low
         ...  
9993    Medium
9994      High
9995       Low
9997    Medium
9999    Medium
Name: mid_category_2ndway, Length: 8596, dtype: str

In [38]:
readable = data.query("`Total Spent` > 2 and Item == 'Coffee'")

readable

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,mid_category,mid_category_2ndway
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,Low,Low
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Low,Low
39,TXN_6688524,Coffee,4.0,2.0,8.0,Unknown,Unknown,2023-06-29,Medium,Medium
80,TXN_3748616,Coffee,2.0,2.0,4.0,Credit Card,In-store,2023-12-09,Low,Low
93,TXN_3011323,Coffee,3.0,2.0,6.0,Credit Card,Unknown,2023-03-07,Medium,Medium
...,...,...,...,...,...,...,...,...,...,...
9966,TXN_6494265,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-05-24,Low,Low
9969,TXN_9954652,Coffee,2.0,2.0,4.0,Cash,In-store,2023-09-12,Low,Low
9990,TXN_1538510,Coffee,5.0,2.0,10.0,Digital Wallet,Unknown,2023-05-22,Medium,Medium
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30,Low,Low


In [40]:
summary  = data.groupby("Item").agg(
    revenue = ("Total Spent", "sum"),
    average = ("Total Spent", "mean"),
    count = ("Total Spent", "count"),
    total_qty = ("Quantity", "sum"),
)

In [41]:
summary

,revenue,average,count,total_qty
Item,,,,
Cake,9009.0,9.109201,989,3003.0
Coffee,6210.0,6.136364,1012,3105.0
Cookie,2775.0,2.987083,929,2775.0
Juice,9108.0,8.929412,1020,3036.0
Salad,14890.0,15.116751,985,2978.0
Sandwich,11680.0,12.053664,969,2920.0
Smoothie,11608.0,12.206099,951,2902.0
Tea,4179.0,4.552288,918,2786.0
Unknown,7231.5,8.786756,823,2458.0


In [44]:
data["month"] = data["Transaction Date"].dt.month
monthly = data.groupby("month")["Total Spent"].sum()
monthly

month
1     6452.5
2     6055.0
3     6482.0
4     6485.5
5     6191.0
6     6678.0
7     6366.5
8     6453.0
9     6209.5
10    6512.0
11    6418.0
12    6387.5
Name: Total Spent, dtype: float64

In [45]:
# Pivot tables

grid = data.pivot_table(
    index = "Item",
    columns = "Payment Method",
    values= "Total Spent",
    aggfunc= "count",
)

grid

Payment Method,Cash,Credit Card,Digital Wallet,Unknown
Item,,,,
Cake,244,222,216,307
Coffee,203,211,235,363
Cookie,215,215,216,283
Juice,236,239,253,292
Salad,249,248,206,282
Sandwich,214,218,219,318
Smoothie,207,215,222,307
Tea,206,193,217,302
Unknown,183,194,196,250


In [ ]:
# Work on a copy for reshaping and feature engineering.
df = data.copy()

In [63]:
g = df.groupby(["Item", "Payment Method"])["Total Spent"].sum()
g

Item      Payment Method
Cake      Cash              2229.0
          Credit Card       2064.0
          Digital Wallet    2007.0
          Unknown           2709.0
Coffee    Cash              1262.0
          Credit Card       1252.0
          Digital Wallet    1414.0
          Unknown           2282.0
Cookie    Cash               656.0
          Credit Card        623.0
          Digital Wallet     638.0
          Unknown            858.0
Juice     Cash              1995.0
          Credit Card       2181.0
          Digital Wallet    2238.0
          Unknown           2694.0
Salad     Cash              3750.0
          Credit Card       3635.0
          Digital Wallet    3175.0
          Unknown           4330.0
Sandwich  Cash              2640.0
          Credit Card       2564.0
          Digital Wallet    2724.0
          Unknown           3752.0
Smoothie  Cash              2652.0
          Credit Card       2612.0
          Digital Wallet    2704.0
          Unknown           36

In [64]:
g.loc[("Coffee","Cash")]

np.float64(1262.0)

In [66]:
g.xs("Cash", level = "Payment Method")

Item
Cake        2229.0
Coffee      1262.0
Cookie       656.0
Juice       1995.0
Salad       3750.0
Sandwich    2640.0
Smoothie    2652.0
Tea          937.5
Unknown     1617.0
Name: Total Spent, dtype: float64

In [67]:
# # Tier 1
# df.iloc[:5]            # 1. first 5 rows
# df.iloc[1]             # 2. 2nd row
# df.iloc[-1]            # 3. last row
# df.iloc[-3:]           # 4. last 3 rows
# df.iloc[[1, 2]]        # 5. rows at positions 1 and 2 (list of positions)
# df.iloc[0:10:2]        # 6. every other row of first 10 (start:stop:step)

# # Tier 2
# df.iloc[1, 2]          # 7. 2nd row, 3rd column → a single cell (the Quantity)
# df.iloc[:5, :3]        # 8. first 5 rows, first 3 columns
# df.iloc[:, 2]          # 9. entire 3rd column (Quantity) as a Series
# df.iloc[9, 1]          # 10. 10th row's Item (position 9, column 1)
# df.iloc[1:4, [1, 4]]   # 11. rows 1–3, columns Item(1) & Total Spent(4)

# # Tier 3
# df.loc[:4]             # 12. labels 0..4 → 6 ROWS (end INCLUDED)
# df.iloc[:4]            #     positions 0..3 → 4 ROWS (end excluded)
# df.loc[2:5]            # 13. labels 2,3,4,5 → 4 rows (inclusive)
# df.iloc[2:5]           #     positions 2,3,4 → 3 rows (exclusive)
# df2 = df.set_index("Transaction ID")
# df2.loc["TXN_4977031"] # 14. row by its ID label
# df.loc[0:2, "Item":"Total Spent"]   # 15. label slices, BOTH ends inclusive

# # Tier 4
# df.loc[df["Quantity"] >= 4, ["Item", "Quantity"]]   # 16. mask + columns, one call
# df.iloc[-1, -1]        # 17. last row, last column

## Reshape and Combine
Join item costs to transactions, calculate profit, and compare revenue across payment methods.

In [70]:
item_info = pd.DataFrame({
    "Item":     ["Coffee","Tea","Juice","Smoothie","Cake","Cookie","Sandwich","Salad"],
    "Category": ["Drink","Drink","Drink","Drink","Food","Food","Food","Food"],
    "Cost":     [0.80, 0.50, 1.20, 1.50, 1.00, 0.40, 1.80, 2.00],
})


In [71]:
# Merge the 2 dataframes
m = df.merge(item_info, on = "Item", how = 'left')

In [72]:
print("rows before:", df.shape[0], "| rows after:", m.shape[0])
print(m["Category"].value_counts(dropna=False))

rows before: 8596 | rows after: 8596
Category
Drink    3901
Food     3872
NaN       823
Name: count, dtype: int64


In [73]:
# Feature engineering


m["Profit"] = m["Total Spent"] - (m["Quantity"] * m["Cost"])

In [74]:
m[["Item","Quantity","Total Spent","Cost","Profit"]].head()


,Item,Quantity,Total Spent,Cost,Profit
0,Coffee,2.0,4.0,0.8,2.4
1,Cake,4.0,12.0,1.0,8.0
2,Cookie,4.0,4.0,0.4,2.4
3,Salad,2.0,10.0,2.0,6.0
4,Coffee,2.0,4.0,0.8,2.4


In [75]:
m.groupby("Category").agg(
    revenue = ("Total Spent", "sum"),
    profit = ("Profit", "sum"),
    orders = ("Transaction ID", "count")
)

,revenue,profit,orders
Category,,,
Drink,31105.0,19231.8,3901
Food,38354.0,23029.0,3872


In [78]:
wide = (m.pivot_table(index="Item", columns="Payment Method", values="Total Spent", aggfunc="sum").reset_index())


In [79]:
wide

Payment Method,Item,Cash,Credit Card,Digital Wallet,Unknown
0,Cake,2229.0,2064.0,2007.0,2709.0
1,Coffee,1262.0,1252.0,1414.0,2282.0
2,Cookie,656.0,623.0,638.0,858.0
3,Juice,1995.0,2181.0,2238.0,2694.0
4,Salad,3750.0,3635.0,3175.0,4330.0
5,Sandwich,2640.0,2564.0,2724.0,3752.0
6,Smoothie,2652.0,2612.0,2704.0,3640.0
7,Tea,937.5,903.0,985.5,1353.0
8,Unknown,1617.0,1716.5,1825.5,2072.5


In [80]:
long = wide.melt(id_vars="Item", var_name="Payment Method", value_name="Revenue")


In [81]:
long = long.sort_values(by="Revenue", ascending=False)
long

,Item,Payment Method,Revenue
31,Salad,Unknown,4330.0
32,Sandwich,Unknown,3752.0
4,Salad,Cash,3750.0
33,Smoothie,Unknown,3640.0
13,Salad,Credit Card,3635.0
22,Salad,Digital Wallet,3175.0
23,Sandwich,Digital Wallet,2724.0
27,Cake,Unknown,2709.0
24,Smoothie,Digital Wallet,2704.0
30,Juice,Unknown,2694.0


In [82]:
back = long.pivot(index = "Item", columns = "Payment Method", values = "Revenue")

In [83]:
back

Payment Method,Cash,Credit Card,Digital Wallet,Unknown
Item,,,,
Cake,2229.0,2064.0,2007.0,2709.0
Coffee,1262.0,1252.0,1414.0,2282.0
Cookie,656.0,623.0,638.0,858.0
Juice,1995.0,2181.0,2238.0,2694.0
Salad,3750.0,3635.0,3175.0,4330.0
Sandwich,2640.0,2564.0,2724.0,3752.0
Smoothie,2652.0,2612.0,2704.0,3640.0
Tea,937.5,903.0,985.5,1353.0
Unknown,1617.0,1716.5,1825.5,2072.5


In [88]:
df2["Item"].value_counts()

Item
Juice       1020
Coffee      1012
Cake         989
Salad        985
Sandwich     969
Smoothie     951
Cookie       929
Tea          918
Unknown      823
Name: count, dtype: int64

In [89]:
cat = {"Juice": "Drink", "Coffee": "Drink", "Tea": "Drink" , "Smoothie": "Drink", "Cake": "Food", "Cookie": "Food", "Sandwich": "Food", "Salad": "Food"}

df2["Category"] = df2["Item"].map(cat) 

In [90]:
df2["Category"].value_counts()

Category
Drink    3901
Food     3872
Name: count, dtype: int64

In [91]:
df2

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,mid_category,mid_category_2ndway,month,Revenue,Revenue2,isbig,Category
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,Low,Low,9,4.0,4.0,False,Drink
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,Medium,Medium,5,12.0,12.0,False,Food
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,Low,Low,7,4.0,4.0,False,Food
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27,Medium,Medium,4,10.0,10.0,False,Food
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Low,Low,6,4.0,4.0,False,Drink
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8591,TXN_4766549,Smoothie,2.0,4.0,8.0,Cash,Unknown,2023-10-20,Medium,Medium,10,8.0,8.0,False,Drink
8592,TXN_7851634,Unknown,4.0,4.0,16.0,Unknown,Unknown,2023-01-08,High,High,1,16.0,16.0,False,NaN
8593,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30,Low,Low,8,4.0,4.0,False,Drink
8594,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02,Medium,Medium,3,8.0,8.0,False,Drink


In [92]:
df2["Total Spent Level"] = np.where(df2["Total Spent"] > 15, "high", np.where(df2["Total Spent"] > 10, "medium", "low"))

In [93]:
df2["Total Spent Level"].value_counts()

Total Spent Level
low       5782
medium    1500
high      1314
Name: count, dtype: int64

In [94]:
df3 = df2.copy()

In [95]:
conditions =  [df3["Total Spent"] > 15, df3["Total Spent"] > 10, df3["Total Spent"] <=10]
choices = ["high", "medium", "low"]

df3["Total Spent Level"] = np.select(conditions, choices, default="else")

In [96]:
df3["Total Spent Level"].value_counts()

Total Spent Level
low       5782
medium    1500
high      1314
Name: count, dtype: int64

In [97]:
def margin(row):
    return row["Total Spent"] - (row["Quantity"] * 0.5)


df2["margin"] = df2.apply(margin, axis=1)

In [98]:
df2["margin"]

0        3.0
1       10.0
2        2.0
3        9.0
4        3.0
        ... 
8591     7.0
8592    14.0
8593     3.0
8594     6.0
8595    10.5
Name: margin, Length: 8596, dtype: float64

In [99]:
df3["bin"] = pd.cut(df3["Total Spent"], bins=[0,5,10,25], labels = ["Low", "Medium", "High"])
df3["bin"].value_counts()

bin
Low       3140
High      2814
Medium    2642
Name: count, dtype: int64

In [100]:
df["spend_quartile"] = pd.qcut(df["Total Spent"], q=4,
                               labels=["Q1","Q2","Q3","Q4"])

In [101]:
df["spend_quartile"].value_counts()

spend_quartile
Q1    2505
Q2    2355
Q4    1955
Q3    1781
Name: count, dtype: int64

In [102]:
len(df3)

8596

In [103]:
df4 = df3.copy()

In [ ]:
# Index transactions by date for time-series analysis.
df4 = (
    df.dropna(subset=["Transaction Date"])
    .set_index("Transaction Date")
    .sort_index()
)
len(df4)

8596

In [105]:
df3.loc[pd.to_datetime(df3["Transaction Date"], errors="coerce").dt.year == 2023]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,mid_category,mid_category_2ndway,month,Revenue,Revenue2,isbig,Category,Total Spent Level,bin
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,Low,Low,9,4.0,4.0,False,Drink,low,Low
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,Medium,Medium,5,12.0,12.0,False,Food,medium,High
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,Low,Low,7,4.0,4.0,False,Food,low,Low
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27,Medium,Medium,4,10.0,10.0,False,Food,low,Medium
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Low,Low,6,4.0,4.0,False,Drink,low,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8591,TXN_4766549,Smoothie,2.0,4.0,8.0,Cash,Unknown,2023-10-20,Medium,Medium,10,8.0,8.0,False,Drink,low,Medium
8592,TXN_7851634,Unknown,4.0,4.0,16.0,Unknown,Unknown,2023-01-08,High,High,1,16.0,16.0,False,NaN,high,High
8593,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30,Low,Low,8,4.0,4.0,False,Drink,low,Low
8594,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02,Medium,Medium,3,8.0,8.0,False,Drink,low,Medium


In [106]:
df4

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,mid_category,mid_category_2ndway,month,Revenue,Revenue2,isbig,Category,Total Spent Level,bin
Transaction Date,,,,,,,,,,,,,,,,
2023-01-01,TXN_7367474,Juice,5.0,3.0,15.0,Digital Wallet,Takeaway,Medium,Medium,1,15.0,15.0,False,Drink,medium,High
2023-01-01,TXN_2690222,Unknown,4.0,3.0,12.0,Unknown,Unknown,Medium,Medium,1,12.0,12.0,False,NaN,medium,High
2023-01-01,TXN_6234882,Cookie,4.0,1.0,4.0,Credit Card,Unknown,Low,Low,1,4.0,4.0,False,Food,low,Low
2023-01-01,TXN_4292280,Cake,4.0,3.0,12.0,Cash,Takeaway,Medium,Medium,1,12.0,12.0,False,Food,medium,High
2023-01-01,TXN_3093284,Coffee,4.0,2.0,8.0,Digital Wallet,In-store,Medium,Medium,1,8.0,8.0,False,Drink,low,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31,TXN_4528664,Salad,3.0,5.0,15.0,Cash,In-store,Medium,Medium,12,15.0,15.0,False,Food,medium,High
2023-12-31,TXN_8933459,Juice,1.0,3.0,3.0,Unknown,Unknown,Low,Low,12,3.0,3.0,False,Drink,low,Low
2023-12-31,TXN_2975443,Salad,5.0,5.0,25.0,Unknown,Takeaway,High,High,12,25.0,25.0,True,Food,high,High


In [ ]:
# Summarize weekly revenue for 2023.
sales_2023 = df4.loc["2023"].copy()
sales_2023["Total Spent"].resample("W").sum()

Transaction Date
2023-01-01     135.0
2023-01-08    1512.0
2023-01-15    1512.0
2023-01-22    1295.0
2023-01-29    1494.0
2023-02-05    1503.0
2023-02-12    1486.5
2023-02-19    1557.0
2023-02-26    1524.0
2023-03-05    1385.0
2023-03-12    1319.0
2023-03-19    1709.5
2023-03-26    1492.0
2023-04-02    1591.0
2023-04-09    1480.0
2023-04-16    1477.5
2023-04-23    1611.0
2023-04-30    1391.5
2023-05-07    1384.5
2023-05-14    1442.0
2023-05-21    1440.5
2023-05-28    1366.0
2023-06-04    1358.0
2023-06-11    1540.5
2023-06-18    1673.5
2023-06-25    1476.0
2023-07-02    1589.0
2023-07-09    1478.0
2023-07-16    1397.0
2023-07-23    1509.5
2023-07-30    1393.0
2023-08-06    1470.5
2023-08-13    1469.5
2023-08-20    1488.0
2023-08-27    1424.5
2023-09-03    1319.0
2023-09-10    1688.5
2023-09-17    1395.0
2023-09-24    1473.0
2023-10-01    1319.0
2023-10-08    1500.0
2023-10-15    1296.0
2023-10-22    1602.0
2023-10-29    1432.5
2023-11-05    1321.0
2023-11-12    1634.0
2023-11-19    158

In [ ]:
sales_2023.loc["2023-06"]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,mid_category,mid_category_2ndway,month,Revenue,Revenue2,isbig,Category,Total Spent Level,bin
Transaction Date,,,,,,,,,,,,,,,,,
2023-06-11,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Low,Low,6,4.0,4.0,False,Drink,low,Low
2023-06-01,TXN_9437049,Cookie,5.0,1.0,5.0,Unknown,Takeaway,2023-06-01,Medium,Medium,6,5.0,5.0,False,Food,low,Low
2023-06-10,TXN_3765707,Sandwich,1.0,4.0,4.0,Unknown,Unknown,2023-06-10,Low,Low,6,4.0,4.0,False,Food,low,Low
2023-06-03,TXN_9400181,Sandwich,5.0,4.0,20.0,Cash,In-store,2023-06-03,High,High,6,20.0,20.0,False,Food,high,High
2023-06-02,TXN_1736287,Unknown,5.0,2.0,10.0,Digital Wallet,Unknown,2023-06-02,Medium,Medium,6,10.0,10.0,False,NaN,low,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-06-28,TXN_7439400,Salad,3.0,5.0,15.0,Unknown,Takeaway,2023-06-28,Medium,Medium,6,15.0,15.0,False,Food,medium,High
2023-06-06,TXN_8330240,Sandwich,4.0,4.0,16.0,Digital Wallet,Unknown,2023-06-06,High,High,6,16.0,16.0,False,Food,high,High
2023-06-22,TXN_8922585,Salad,5.0,5.0,25.0,Credit Card,Unknown,2023-06-22,High,High,6,25.0,25.0,True,Food,high,High


In [ ]:
%timeit df["Total Spent"] * 1.2

329 μs ± 69.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%timeit df["Total Spent"].apply(lambda x: x * 1.2)

5.68 ms ± 2.11 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%timeit [x * 1.2 for x in df["Total Spent"]]

1.66 ms ± 389 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
df.memory_usage(deep=True)

Index                     132
Transaction ID         584528
Item                   539474
Quantity                68768
Price Per Unit          68768
Total Spent             68768
Payment Method         565953
Location               555333
Transaction Date        68768
mid_category           530784
mid_category_2ndway    530784
month                   34384
spend_quartile           9004
dtype: int64

## SENIOR SKILLS

In [3]:
os.chdir('C:/Users/lefte/pc applications/Desktop/datasets')

In [7]:
import pandas as pd
import numpy as np
data = pd.read_csv("cafe_sales.csv")

In [8]:
data = data.replace(["ERRORS", "UNKNOWN"],np.nan)

In [10]:
for c in ["Quantity", "Total Spent", "Price Per Unit"]:
    data[c] = pd.to_numeric(data[c],errors = "coerce")


In [12]:
data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], errors = "coerce")

In [14]:
for i in ["Item", "Payment Method", "Location"]:
    data[i] = data[i].astype("category")

In [17]:
print(data.isna().sum())

Transaction ID         0
Item                 677
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      2872
Location            3603
Transaction Date     460
dtype: int64


In [20]:
# Total Spent  - Price Per Unit - Quantity

data["Total Spent"] = data["Total Spent"].fillna(data["Price Per Unit"] * data["Quantity"])
data["Price Per Unit"] = data["Price Per Unit"].fillna(data["Total Spent"] / data["Quantity"])
data["Quantity"] = data["Quantity"].fillna(data["Total Spent"] / data["Price Per Unit"])

In [22]:
tem_info = pd.DataFrame({
    "Item":     ["Coffee","Tea","Juice","Smoothie","Cake","Cookie","Sandwich","Salad"],
    "Category": ["Drink","Drink","Drink","Drink","Food","Food","Food","Food"],
    "Cost":     [0.80,0.50,1.20,1.50,1.00,0.40,1.80,2.00],
})

In [24]:
data = data.merge(tem_info, on = "Item", how = "left")

In [34]:
data["Profit"] = data["Total Spent"] - (data["Quantity"] * data["Cost"])

In [36]:
data["Profit"]

0       0.0
1       4.0
2      -4.0
3       6.0
4       0.0
       ... 
9995    0.0
9996   -3.0
9997    0.0
9998   -3.0
9999    6.0
Name: Profit, Length: 10000, dtype: float64

In [38]:
data["spend_band"] = np.select([data["Total Spent"] < 5, data["Total Spent"] < 10], ["low", "medium"], default="high")

In [41]:
data["month"] = data["Transaction Date"].dt.to_period("M").astype(str)